In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Read the dataset
df = pd.read_csv('icecreamsales.csv')

In [2]:
# Convert the 'Month' column to datetime
df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m-%d')

In [3]:
# Create a time index for regression analysis
df['Time_Index'] = np.arange(len(df)) + 1

In [4]:
# Calculate average sales for each month
monthly_avg_sales = df.groupby(df['Month'].dt.month)['Sales'].mean()

# Calculate the overall average sales
overall_avg_sales = df['Sales'].mean()

# Calculate seasonal indices
seasonal_indices = monthly_avg_sales / overall_avg_sales

In [5]:
seasonal_indices

Month
1     1.327855
2     1.046311
3     1.016322
4     0.937128
5     0.887403
6     0.688991
7     0.811964
8     0.863523
9     0.880373
10    1.046788
11    1.193284
12    1.300058
Name: Sales, dtype: float64

In [6]:
# Apply the seasonal index to deseasonalize the data
df['Deseasonalized_Sales'] = df.apply(lambda row: row['Sales'] / seasonal_indices[row['Month'].month], axis=1)

In [7]:
# Linear Regression on Deseasonalized Data
model = smf.ols(formula='Deseasonalized_Sales ~ Time_Index', data=df).fit()

In [8]:
# Forecast sales for the next year (12 months)
df_forecast = pd.DataFrame({
    'Month': pd.date_range(start=df['Month'].max() + pd.DateOffset(1), periods=12, freq='M'),
    'Time_Index': np.arange(len(df) + 1, len(df) + 13)
})

In [9]:
df_forecast

,Month,Time_Index
0,2022-01-31,37
1,2022-02-28,38
2,2022-03-31,39
3,2022-04-30,40
4,2022-05-31,41
5,2022-06-30,42
6,2022-07-31,43
7,2022-08-31,44
8,2022-09-30,45
9,2022-10-31,46


In [10]:
# Predict using the model
df_forecast['Forecasted_Deseasonalized_Sales'] = model.predict(df_forecast)

In [11]:
df_forecast

,Month,Time_Index,Forecasted_Deseasonalized_Sales
0,2022-01-31,37,295.813038
1,2022-02-28,38,295.196325
2,2022-03-31,39,294.579613
3,2022-04-30,40,293.962900
4,2022-05-31,41,293.346187
5,2022-06-30,42,292.729475
6,2022-07-31,43,292.112762
7,2022-08-31,44,291.496049
8,2022-09-30,45,290.879337
9,2022-10-31,46,290.262624


In [12]:
# Reintroduce seasonality to the forecast
df_forecast['Seasonal_Index'] = df_forecast['Month'].dt.month.apply(lambda x: seasonal_indices[x])
df_forecast['Seasonally_Adjusted_Forecast'] = df_forecast['Forecasted_Deseasonalized_Sales'] * df_forecast['Seasonal_Index']

In [13]:
df_forecast

,Month,Time_Index,Forecasted_Deseasonalized_Sales,Seasonal_Index,Seasonally_Adjusted_Forecast
0,2022-01-31,37,295.813038,1.327855,392.796920
1,2022-02-28,38,295.196325,1.046311,308.867171
2,2022-03-31,39,294.579613,1.016322,299.387706
3,2022-04-30,40,293.962900,0.937128,275.480979
4,2022-05-31,41,293.346187,0.887403,260.316361
5,2022-06-30,42,292.729475,0.688991,201.687961
6,2022-07-31,43,292.112762,0.811964,237.184998
7,2022-08-31,44,291.496049,0.863523,251.713427
8,2022-09-30,45,290.879337,0.880373,256.082173
9,2022-10-31,46,290.262624,1.046788,303.843555


In [14]:
# Display the seasonally adjusted forecast
df_forecast[['Month', 'Seasonally_Adjusted_Forecast']]

,Month,Seasonally_Adjusted_Forecast
0,2022-01-31,392.796920
1,2022-02-28,308.867171
2,2022-03-31,299.387706
3,2022-04-30,275.480979
4,2022-05-31,260.316361
5,2022-06-30,201.687961
6,2022-07-31,237.184998
7,2022-08-31,251.713427
8,2022-09-30,256.082173
9,2022-10-31,303.843555
